In [1]:
import json
import os
import time
import uuid
from pathlib import Path

import boto3
import psycopg
from dotenv import load_dotenv, set_key, unset_key

import sys
sys.path.insert(0, "..")  # the helper is in labs/week3
import week3day4_aws as aws

In [2]:
load_dotenv(override=True)

session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

Working as arn:aws:iam::123456789012:user/undercover in us-east-1


In [3]:
lambda_client = session.client("lambda")
iam = session.client("iam")
logs = session.client("logs")
agentcore_runtime = session.client("bedrock-agentcore")

In [4]:
for name in ["OPENROUTER_API_KEY", "DATABASE_URL", "VITE_SUPABASE_URL", "UNDERCOVER_API_KEY"]:
    print(f"{name}: {'found' if os.getenv(name) else 'missing, please add it to .env'}")

API_KEY = os.environ["UNDERCOVER_API_KEY"]
# Supabase's transaction pooler, which the app on Lambda and the agent on Runtime share
POOLED_DATABASE_URL = os.environ["DATABASE_URL"].replace(":5432/", ":6543/")

OPENROUTER_API_KEY: found
DATABASE_URL: found
VITE_SUPABASE_URL: found
UNDERCOVER_API_KEY: found


In [5]:
!cd .. && npm run build

'npm' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [6]:
app_environment = {
    "DATABASE_URL": POOLED_DATABASE_URL,
    "VITE_SUPABASE_URL": os.environ["VITE_SUPABASE_URL"],
    "UNDERCOVER_API_KEY": API_KEY,
}

app_url = aws.deploy_app(app_environment)
print(app_url)

1/4 Listing the packages the app needs
2/4 Packages for Linux: already installed from this same list, kept (the slow step is skipped)
3/4 Copying our code
4/4 Making the zip
lambda.zip: 39.5 MB (Lambda takes up to 50 MB), packaged in 63 s
Log group /aws/lambda/undercover-app already exists
Role undercover-app-role already exists
Sending lambda.zip to AWS and waiting for undercover-app to be ready
Updated function undercover-app
https://example.lambda-url.us-east-1.on.aws


In [7]:
context_url = aws.deploy_context()
print(context_url)

1/4 Listing the packages the app needs
2/4 Packages for Linux: already installed from this same list, kept (the slow step is skipped)
3/4 Copying our code
4/4 Making the zip
context.zip: 10.6 MB (Lambda takes up to 50 MB), packaged in 9 s
Log group /aws/lambda/undercover-context already exists
Role undercover-context-role already exists
Sending context.zip to AWS and waiting for undercover-context to be ready
Updated function undercover-context
https://example.lambda-url.us-east-1.on.aws


In [8]:
gateway = aws.create_gateway(app_url, context_url, API_KEY)
gateway_url = gateway["gatewayUrl"]
print(gateway_url)

Credential provider undercover-app-key already exists, key updated
Role undercover-gateway-role already exists
Gateway undercover-gateway-ap7hyvigje: READY
Target undercover-app: READY
Target undercover-context: READY
Policy engine undercover_policy_engine-lcendkzwl0: ACTIVE
Gateway undercover-gateway-ap7hyvigje: READY
Policy undercover_allow_app_tools: ACTIVE
Policy undercover_allow_context_tools: ACTIVE
Policy undercover_payment_limit: ACTIVE
https://undercover-gateway-ap7hyvigje.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp


In [9]:
memory_id = aws.create_memory()

Memory undercover_memory-qHEt1f51nG: ACTIVE


In [10]:
LOCAL_ENV = Path("../../../agentcore/.env.local")
LOCAL_ENV.touch()

secrets = {
    "AGENTCORE_CREDENTIAL_OPENROUTER_API_KEY": os.environ["OPENROUTER_API_KEY"],
    "AGENTCORE_CREDENTIAL_UNDERCOVER_API_KEY": API_KEY,
    "AGENTCORE_CREDENTIAL_DATABASE_URL": POOLED_DATABASE_URL,
}
settings = {
    "UNDERCOVER_API": app_url,
    "UNDERCOVER_GATEWAY_URL": gateway_url,
    "UNDERCOVER_MEMORY_ID": memory_id,
}

for name, value in (secrets | settings).items():
    set_key(LOCAL_ENV, name, value)

print(f"Wrote {', '.join(secrets | settings)} to {LOCAL_ENV}")

Wrote AGENTCORE_CREDENTIAL_OPENROUTER_API_KEY, AGENTCORE_CREDENTIAL_UNDERCOVER_API_KEY, AGENTCORE_CREDENTIAL_DATABASE_URL, UNDERCOVER_API, UNDERCOVER_GATEWAY_URL, UNDERCOVER_MEMORY_ID to ..\agentcore\.env.local


In [11]:
runtime = aws.find_runtimes("undercover_default_agent")[0]
runtime_arn = runtime["agentRuntimeArn"]
print(runtime_arn)

arn:aws:bedrock-agentcore:us-east-1:123456789012:runtime/undercover_default_agent-I1EP5yBFbb


In [13]:
def call_agent(payload):
    """Send a payload to the agent on Runtime in a new session, as the app does."""
    started = time.time()
    response = agentcore_runtime.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn, runtimeSessionId=f"lab-{uuid.uuid4()}", payload=json.dumps(payload)
    )
    reply = json.loads(response["response"].read())
    print(f"{time.time() - started:.1f} seconds: {reply}")


call_agent({"operation": "health"})

3.8 seconds: {'healthy': True}


In [14]:
app_environment["UNDERCOVER_AGENT_RUNTIME"] = runtime_arn

lambda_client.update_function_configuration(FunctionName=aws.APP_FUNCTION, Environment={"Variables": app_environment})
lambda_client.get_waiter("function_updated_v2").wait(FunctionName=aws.APP_FUNCTION)
print(f"{aws.APP_FUNCTION} starts the agent's work on {runtime_arn}")

undercover-app starts the agent's work on arn:aws:bedrock-agentcore:us-east-1:123456789012:runtime/undercover_default_agent-I1EP5yBFbb


In [20]:
!cd .. && npm run seed

'npm' n�o � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


In [16]:
print(app_url)

https://example.lambda-url.us-east-1.on.aws


In [17]:
COUNT_CLAIMS = "SELECT status, count(*) FROM undercover.claims WHERE evaluation_id IS NULL GROUP BY status ORDER BY status"

with psycopg.connect(os.environ["DATABASE_URL"], autocommit=True) as connection:
    for check in range(80):
        counts = dict(connection.execute(COUNT_CLAIMS).fetchall())
        print(", ".join(f"{status} {number}" for status, number in counts.items()))
        if not counts.keys() & {"waiting", "in_progress"}:
            break
        time.sleep(15)

awaiting_approval 3, decided 23


In [18]:
invoke_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "bedrock-agentcore:InvokeAgentRuntime",
            "Resource": [runtime_arn, f"{runtime_arn}/runtime-endpoint/*"],
        }
    ],
}

iam.put_role_policy(RoleName=aws.APP_ROLE_NAME, PolicyName="undercover-app-runtime", PolicyDocument=json.dumps(invoke_policy))
print(f"{aws.APP_ROLE_NAME} may invoke the agent's runtime")

undercover-app-role may invoke the agent's runtime


In [19]:
app_environment["UNDERCOVER_AGENT_RUNTIME"] = runtime_arn

lambda_client.update_function_configuration(FunctionName=aws.APP_FUNCTION, Environment={"Variables": app_environment})
lambda_client.get_waiter("function_updated_v2").wait(FunctionName=aws.APP_FUNCTION)
print(f"{aws.APP_FUNCTION} starts the agent's work on {runtime_arn}")

undercover-app starts the agent's work on arn:aws:bedrock-agentcore:us-east-1:123456789012:runtime/undercover_default_agent-I1EP5yBFbb


In [21]:
for name in settings:
    unset_key(LOCAL_ENV, name)

print(f"Took {', '.join(settings)} out of {LOCAL_ENV}")

Took UNDERCOVER_API, UNDERCOVER_GATEWAY_URL, UNDERCOVER_MEMORY_ID out of ..\agentcore\.env.local


In [22]:
LOG_GROUPS = "/aws/bedrock-agentcore/runtimes/undercover_default_agent"

for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=LOG_GROUPS):
    for group in page["logGroups"]:
        logs.delete_log_group(logGroupName=group["logGroupName"])
        print(f"Deleted log group {group['logGroupName']}")

Deleted log group /aws/bedrock-agentcore/runtimes/undercover_default_agent-I1EP5yBFbb-DEFAULT


In [23]:
aws.tear_down()

Deleted target undercover-context
Deleted target undercover-app
Deleted gateway undercover-gateway
Deleted policy undercover_allow_app_tools
Deleted policy undercover_allow_context_tools
Deleted policy undercover_payment_limit
Deleted policy engine undercover_policy_engine
Deleted credential provider undercover-app-key
Deleted memory undercover_memory-qHEt1f51nG
Deleted role undercover-gateway-role


In [24]:
aws.delete_bootstrap()

Deleted the CDKToolkit stack
Deleted the bucket cdk-hnb659fds-assets-123456789012-us-east-1


In [25]:
DELETE_THE_FUNCTIONS = False

if DELETE_THE_FUNCTIONS:
    aws.delete_functions()

In [26]:
time.sleep(10)
aws.show_resources()

Lambda functions:   undercover-context, undercover-app
Gateways:           none
Policy engines:     none
API key providers:  none
Memories:           none
Runtimes:           undercover_default_agent
Stacks:             AgentCore-undercover-default
IAM roles:          undercover-app-role, undercover-context-role
